In [42]:
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, classification_report

df = pd.read_csv("credit_card_data-headers.txt", sep="\t")

df.shape #inspecting no of rows and cols

df.head()


,A1,A2,A3,A8,A9,A10,A11,A12,A14,A15,R1
0,1,30.83,0.000,1.25,1,0,1,1,202,0,1
1,0,58.67,4.460,3.04,1,0,6,1,43,560,1
2,0,24.50,0.500,1.50,1,1,0,1,280,824,1
3,1,27.83,1.540,3.75,1,0,5,0,100,3,1
4,1,20.17,5.625,1.71,1,1,0,1,120,0,1


In [43]:
#splitting the data into predictors and response variables

x = df.iloc[:,:-1] #predictors
y = df.iloc[:,-1] #response



In [44]:
#scaling the predictors variables

scaler = StandardScaler()

x_scaled = scaler.fit_transform(x)


In [45]:
#training the model

knn_model = KNeighborsClassifier(n_neighbors=5) #initializing the model

knn_model.fit(x_scaled, y) #actually training the model


,"n_neighbors n_neighbors: int, default=5Number of neighbors to use by default for :meth:`kneighbors` queries.",5
,"weights weights: {'uniform', 'distance'}, callable or None, default='uniform'Weight function used in prediction. Possible values:- 'uniform' : uniform weights. All points in each neighborhood are weighted equally.- 'distance' : weight points by the inverse of their distance. in this case, closer neighbors of a query point will have a greater influence than neighbors which are further away.- [callable] : a user-defined function which accepts an array of distances, and returns an array of the same shape containing the weights.Refer to the example entitled:ref:`sphx_glr_auto_examples_neighbors_plot_classification.py`showing the impact of the `weights` parameter on the decisionboundary.",'uniform'
,"algorithm algorithm: {'auto', 'ball_tree', 'kd_tree', 'brute'}, default='auto'Algorithm used to compute the nearest neighbors:- 'ball_tree' will use :class:`BallTree`- 'kd_tree' will use :class:`KDTree`- 'brute' will use a brute-force search.- 'auto' will attempt to decide the most appropriate algorithm based on the values passed to :meth:`fit` method.Note: fitting on sparse input will override the setting ofthis parameter, using brute force.",'auto'
,"leaf_size leaf_size: int, default=30Leaf size passed to BallTree or KDTree. This can affect thespeed of the construction and query, as well as the memoryrequired to store the tree. The optimal value depends on thenature of the problem.",30
,"p p: float, default=2Power parameter for the Minkowski metric. When p = 1, this is equivalentto using manhattan_distance (l1), and euclidean_distance (l2) for p = 2.For arbitrary p, minkowski_distance (l_p) is used. This parameter is expectedto be positive.",2
,"metric metric: str or callable, default='minkowski'Metric to use for distance computation. Default is ""minkowski"", whichresults in the standard Euclidean distance when p = 2. See thedocumentation of `scipy.spatial.distance`_ andthe metrics listed in:class:`~sklearn.metrics.pairwise.distance_metrics` for valid metricvalues.If metric is ""precomputed"", X is assumed to be a distance matrix andmust be square during fit. X may be a :term:`sparse graph`, in whichcase only ""nonzero"" elements may be considered neighbors.If metric is a callable function, it takes two arrays representing 1Dvectors as inputs and must return one value indicating the distancebetween those vectors. This works for Scipy's metrics, but is lessefficient than passing the metric name as a string.",'minkowski'
,"metric_params metric_params: dict, default=NoneAdditional keyword arguments for the metric function.",None
,"n_jobs n_jobs: int, default=NoneThe number of parallel jobs to run for neighbors search.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details.Doesn't affect :meth:`fit` method.",None


In [46]:
#asking a trained model to classify observations

pred = knn_model.predict(x_scaled)#predict on the data that I give you

pd.Series(pred).value_counts() #checks how many 1's and 0's it predicted


0    352
1    302
Name: count, dtype: int64

In [47]:
#evaluating the model's accuracy

accuracy = accuracy_score(y, pred)
round(accuracy * 100, 2 )

89.3

In [53]:
# #checking models accuracy for different values of k nearest neighbours

# for i in range(len(x_scaled)):
#     train_indicies = [j for j in range(len(x_scaled)) if j != i]

# k_values = [5,7,10,12,15,20]
# different_k_accuracies = []

# for k in k_values:
#     knn_model = KNeighborsClassifier(n_neighbors=k)
#     knn_model.fit(x_scaled, y)
#     pred = knn_model.predict(x_scaled)
#     accuracy = accuracy_score(y, pred)

#     different_k_accuracies.append({
#         "K value": k,
#         "Accuracy": accuracy
#     })

# different_k_accuracies_df = pd.DataFrame(different_k_accuracies)
# different_k_accuracies_df

# checking model accuracy for different values of k nearest neighbours

k_values = [1,5, 7, 10, 12, 15, 20,22,25]
different_k_accuracies = []

for k in k_values:

    predictions = []

    for i in range(len(x_scaled)):

        # Leave observation i out of the training data
        train_indices = [j for j in range(len(x_scaled)) if j != i]

        # Create training data
        X_train = x_scaled[train_indices]
        y_train = y.iloc[train_indices]

        # Create test data: only observation i
        X_test = x_scaled[i].reshape(1, -1)

        # Create and train the KNN model
        knn_model = KNeighborsClassifier(n_neighbors=k)
        knn_model.fit(X_train, y_train)

        # Predict the observation we left out
        prediction = knn_model.predict(X_test)

        # Save the prediction
        predictions.append(prediction[0])

    # Compare all predictions with the actual responses
    accuracy = accuracy_score(y, predictions)

    different_k_accuracies.append({
        "K value": k,
        "Accuracy": accuracy
    })

different_k_accuracies_df = pd.DataFrame(different_k_accuracies)

different_k_accuracies_df

,K value,Accuracy
0,1,0.814985
1,5,0.845566
2,7,0.845566
3,10,0.840979
4,12,0.833333
5,15,0.824159
6,20,0.837920
7,22,0.853211
8,25,0.833333
